# Waste Type Identification — Reject option (rigetto verso `undifferentiated`)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

BASE = Path('/content/drive/MyDrive/2026_MLinf_gr41/Waste-Project')
DRIVE_DATASET_DIR = BASE / 'dataset'
DATASET_DIR = Path('/content/dataset_local')
SPLIT_CSV = BASE / 'splits' / 'split.csv'
MODELS_DIR = BASE / 'models'
RESULTS_DIR = BASE / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE = 224
NUM_CLASSES = 8
SEED = 1234

CLASS_NAMES = ['battery','clothing','glass','metal','organic','papery','plastic','undifferentiated']
UNDIFFERENTIATED_LABEL = CLASS_NAMES.index('undifferentiated')                  # 7
TARGET_CLASSES = ['metal', 'plastic']
WATCH_CLASSES = ['clothing', 'undifferentiated']
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

FAMILIES = ['geometric', 'acquisition', 'background', 'resolution']
INTENSITIES = ['mild', 'moderate']

# Modello finale
FINAL_ARCH = 'regnety16gf'
FINAL_WEIGHTS_PATH = MODELS_DIR / 'regnety16gf_acq_mild.pth'

# Soglie pre-registrate
TAU_CANDIDATES = [0.5, 0.7]

# Floor pre-registrato
NOISE_FLOOR = 0.01

Mounted at /content/drive


In [ ]:
import io, os, random, time, shutil
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image, ImageEnhance, ImageFilter
from tqdm.auto import tqdm
from sklearn.metrics import balanced_accuracy_score, recall_score

def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device, '|', torch.cuda.get_device_name(0) if device.type=='cuda' else 'CPU')

if not DATASET_DIR.exists():
    print('Copio il dataset in locale...')
    t0 = time.time(); shutil.copytree(DRIVE_DATASET_DIR, DATASET_DIR)
    print(f'Fatto in {time.time()-t0:.0f}s')
else:
    print('Copia locale gia presente.')

Device: cuda | Tesla T4
Copio il dataset in locale...
Fatto in 631s


In [ ]:
df_all = pd.read_csv(SPLIT_CSV)
df_val = df_all[df_all['split'] == 'val'].reset_index(drop=True).copy()
print(f"Val: {len(df_val)}")

preprocess = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

Val: 3102


In [ ]:
def _rng(idx, family, intensity):
    fam_id = {'geometric':1, 'acquisition':2, 'background':3, 'resolution':4}[family]
    int_id = {'mild':1, 'moderate':2}[intensity]
    return np.random.RandomState((SEED * 100003 + idx * 97 + fam_id * 13 + int_id) % (2**32))

def perturb_geometric(img, idx, intensity):
    r = _rng(idx, 'geometric', intensity)
    max_rot, min_scale = (8.0, 0.85) if intensity == 'mild' else (15.0, 0.60)
    W, H = img.size
    ang = r.uniform(-max_rot, max_rot)
    img = img.rotate(ang, resample=Image.BILINEAR, expand=False, fillcolor=(255, 255, 255))
    scale = r.uniform(min_scale, 1.0)
    cw, ch = max(1, int(W * np.sqrt(scale))), max(1, int(H * np.sqrt(scale)))
    x0 = r.randint(0, max(1, W - cw + 1)); y0 = r.randint(0, max(1, H - ch + 1))
    return img.crop((x0, y0, x0 + cw, y0 + ch)).resize((W, H), Image.BILINEAR)

def perturb_acquisition(img, idx, intensity):
    r = _rng(idx, 'acquisition', intensity)
    blur, q, jit, noise = (0.6, 70, 0.10, 4.0) if intensity == 'mild' else (1.2, 40, 0.20, 10.0)
    img = img.filter(ImageFilter.GaussianBlur(radius=blur * r.uniform(0.7, 1.3)))
    for Enh in (ImageEnhance.Brightness, ImageEnhance.Contrast, ImageEnhance.Color):
        img = Enh(img).enhance(1.0 + r.uniform(-jit, jit))
    buf = io.BytesIO(); img.save(buf, format='JPEG', quality=int(q)); buf.seek(0)
    img = Image.open(buf).convert('RGB')
    arr = np.asarray(img).astype(np.float32) + r.normal(0, noise, (img.size[1], img.size[0], 3))
    return Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))

def perturb_resolution(img, idx, intensity):
    r = _rng(idx, 'resolution', intensity)
    f = (0.5 if intensity == 'mild' else 0.34) * r.uniform(0.9, 1.1)
    W, H = img.size
    small = img.resize((max(1, int(W * f)), max(1, int(H * f))), Image.BILINEAR)
    return small.resize((W, H), Image.BILINEAR)

def perturb_background(img, idx, intensity):
    keep = 0.80 if intensity == 'mild' else 0.65
    W, H = img.size
    cw, ch = int(W * keep), int(H * keep)
    x0, y0 = (W - cw) // 2, (H - ch) // 2
    canvas = Image.new('RGB', (W, H), (255, 255, 255))
    canvas.paste(img.crop((x0, y0, x0 + cw, y0 + ch)), (x0, y0))
    return canvas

PERTURB = {'geometric': perturb_geometric, 'acquisition': perturb_acquisition,
           'background': perturb_background, 'resolution': perturb_resolution}

In [ ]:
def build_backbone(name):
    if name == 'resnet18':
        m = models.resnet18(weights='DEFAULT'); m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)
    elif name == 'regnety16gf':
        m = models.regnet_y_1_6gf(weights='DEFAULT'); m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)
    elif name == 'effv2s':
        m = models.efficientnet_v2_s(weights='DEFAULT'); m.classifier[1] = nn.Linear(m.classifier[1].in_features, NUM_CLASSES)
    elif name == 'convnext_tiny':
        m = models.convnext_tiny(weights='DEFAULT'); m.classifier[2] = nn.Linear(m.classifier[2].in_features, NUM_CLASSES)
    else:
        raise ValueError(name)
    return m

class ValDataset(Dataset):
    def __init__(self, df, family=None, intensity=None):
        self.items = list(zip(df['filepath'].tolist(), df['label'].tolist()))
        self.family, self.intensity = family, intensity
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        fp, lab = self.items[i]
        img = Image.open(DATASET_DIR / fp).convert('RGB')
        if self.family is not None:
            img = PERTURB[self.family](img, i, self.intensity)
        return preprocess(img), lab

In [ ]:
# Valutazione con TTA + reject opzionale

@torch.no_grad()
def evaluate_reject(model, df, family=None, intensity=None, tau=None, bs=64, use_tta=True):
    """
    tau=None  -> baseline, nessun reject
    tau=float -> se la confidenza massima (dopo TTA) è < tau, la predizione viene ridiretta a UNDIFFERENTIATED_LABEL
    Ritorna: balanced accuracy, dict per-classe TPR, frazione di campioni rigettati
    """
    model.eval()
    dl = DataLoader(ValDataset(df, family, intensity), batch_size=bs, shuffle=False, num_workers=2)
    ys, preds, maxconf = [], [], []
    for x, y in dl:
        x = x.to(device)
        if use_tta:
            flipped = torch.flip(x, dims=[3])
            both = torch.cat([x, flipped], dim=0)
            probs = F.softmax(model(both), dim=1)
            B = x.shape[0]
            probs = 0.5 * (probs[:B] + probs[B:])
        else:
            probs = F.softmax(model(x), dim=1)
        conf, pred = probs.max(dim=1)
        if tau is not None:
            pred = pred.clone()
            pred[conf < tau] = UNDIFFERENTIATED_LABEL
        preds.append(pred.cpu().numpy())
        maxconf.append(conf.cpu().numpy())
        ys.append(np.asarray(y))
    y = np.concatenate(ys); p = np.concatenate(preds); c = np.concatenate(maxconf)
    bal = balanced_accuracy_score(y, p)
    tpr = recall_score(y, p, labels=list(range(NUM_CLASSES)), average=None, zero_division=0)
    frac_rejected = float((c < tau).mean()) if tau is not None else 0.0
    return bal, dict(zip(CLASS_NAMES, tpr)), frac_rejected

In [ ]:
# Carica il modello finale (nessun retraining)

assert FINAL_WEIGHTS_PATH.exists(), f"Pesi mancanti: {FINAL_WEIGHTS_PATH}"
model = build_backbone(FINAL_ARCH).to(device)
model.load_state_dict(torch.load(FINAL_WEIGHTS_PATH, map_location='cpu'))
model.eval()
print(f"Modello caricato: {FINAL_WEIGHTS_PATH.name} (TTA flip attivo)")

In [ ]:
# Loop principale: baseline (no-reject) + un tau alla volta

rows = []
variants = [None] + TAU_CANDIDATES                                              # None = baseline, poi i due tau pre-registrati

for tau in variants:
    tag = 'no_reject' if tau is None else f'tau_{tau}'
    print(f"\n=== Variante: {tag} ===")

    clean_bal, clean_tpr, clean_frac = evaluate_reject(model, df_val, tau=tau)
    print(f"  clean-val balAcc = {clean_bal:.4f} (metal {clean_tpr['metal']:.3f}, "
          f"plastic {clean_tpr['plastic']:.3f}, undiff. {clean_tpr['undifferentiated']:.3f}, "
          f"clothing {clean_tpr['clothing']:.3f}) | rigettati {clean_frac:.1%}")

    fam_int = {}
    for fam in FAMILIES:
        for inten in INTENSITIES:
            bal, tpr, frac = evaluate_reject(model, df_val, fam, inten, tau=tau)
            fam_int[(fam, inten)] = (bal, tpr, frac)

    def fam_mean(inten):
        return float(np.mean([fam_int[(f, inten)][0] for f in FAMILIES]))
    hard_mild, hard_mod = fam_mean('mild'), fam_mean('moderate')
    hard_mean = (hard_mild + hard_mod) / 2
    acq_mean = (fam_int[('acquisition','mild')][0] + fam_int[('acquisition','moderate')][0]) / 2
    acq_mod_tpr = fam_int[('acquisition','moderate')][1]
    acq_mod_frac = fam_int[('acquisition','moderate')][2]

    print(f"  hard_mean = {hard_mean:.4f} | acquisition_mean = {acq_mean:.4f} | "
          f"rigettati (acquisition, moderate) = {acq_mod_frac:.1%}")

    rows.append({
        'variant': tag, 'tau': tau,
        'clean_val': round(clean_bal, 4),
        'hard_mild': round(hard_mild, 4), 'hard_moderate': round(hard_mod, 4),
        'hard_mean': round(hard_mean, 4), 'acquisition_mean': round(acq_mean, 4),
        'metal_tpr_acq_mod': round(acq_mod_tpr['metal'], 4),
        'plastic_tpr_acq_mod': round(acq_mod_tpr['plastic'], 4),
        'clothing_tpr_clean': round(clean_tpr['clothing'], 4),
        'undifferentiated_tpr_clean': round(clean_tpr['undifferentiated'], 4),
        'clothing_tpr_acq_mod': round(acq_mod_tpr['clothing'], 4),
        'undifferentiated_tpr_acq_mod': round(acq_mod_tpr['undifferentiated'], 4),
        'frac_rejected_clean': round(clean_frac, 4),
        'frac_rejected_acq_mod': round(acq_mod_frac, 4),
    })

result_df = pd.DataFrame(rows)
result_df.to_csv(RESULTS_DIR / 'reject_option_hardval.csv', index=False)
print("\nSalvato: reject_option_hardval.csv\n")
result_df


=== Variante: no_reject ===
  clean-val balAcc = 0.9800 (metal 0.968, plastic 0.954, undiff. 0.978, clothing 0.999) | rigettati 0.0%
  hard_mean = 0.9724 | acquisition_mean = 0.9746 | rigettati (acquisition, moderate) = 0.0%

=== Variante: tau_0.5 ===
  clean-val balAcc = 0.9800 (metal 0.968, plastic 0.954, undiff. 0.978, clothing 0.999) | rigettati 0.2%
  hard_mean = 0.9701 | acquisition_mean = 0.9726 | rigettati (acquisition, moderate) = 0.5%

=== Variante: tau_0.7 ===
  clean-val balAcc = 0.9738 (metal 0.968, plastic 0.936, undiff. 0.978, clothing 0.998) | rigettati 1.0%
  hard_mean = 0.9601 | acquisition_mean = 0.9614 | rigettati (acquisition, moderate) = 2.6%

Salvato: reject_option_hardval.csv



,variant,tau,clean_val,hard_mild,hard_moderate,hard_mean,acquisition_mean,metal_tpr_acq_mod,plastic_tpr_acq_mod,clothing_tpr_clean,undifferentiated_tpr_clean,clothing_tpr_acq_mod,undifferentiated_tpr_acq_mod,frac_rejected_clean,frac_rejected_acq_mod
0,no_reject,NaN,0.9800,0.9785,0.9662,0.9724,0.9746,0.9481,0.9364,0.9993,0.9784,0.9979,0.9496,0.0000,0.0000
1,tau_0.5,0.5,0.9800,0.9765,0.9636,0.9701,0.9726,0.9351,0.9191,0.9986,0.9784,0.9973,0.9640,0.0023,0.0045
2,tau_0.7,0.7,0.9738,0.9701,0.9501,0.9601,0.9614,0.8961,0.8844,0.9979,0.9784,0.9938,0.9712,0.0097,0.0258


In [ ]:
# Verdetto secondo il floor pre-registrato

baseline = result_df[result_df['tau'].isna()].iloc[0]
print(f"Baseline (no reject): hard_mean={baseline['hard_mean']:.4f} | "
      f"acquisition_mean={baseline['acquisition_mean']:.4f}\n")

for _, r in result_df[result_df['tau'].notna()].iterrows():
    d_hard = r['hard_mean'] - baseline['hard_mean']
    d_acq = r['acquisition_mean'] - baseline['acquisition_mean']
    passes = (d_hard >= -NOISE_FLOOR) and (d_acq >= -NOISE_FLOOR)
    verdict = 'PASS (entro il floor)' if passes else 'FAIL (peggiora oltre il floor pre-registrato)'
    print(f"tau={r['tau']}: Δhard_mean={d_hard:+.4f} | Δacquisition_mean={d_acq:+.4f} "
          f"| rigettati(acq,mod)={r['frac_rejected_acq_mod']:.1%} -> {verdict}")
    print(f"   clothing TPR (attrattore, acq. moderate): "
          f"{baseline['clothing_tpr_acq_mod']:.3f} -> {r['clothing_tpr_acq_mod']:.3f} "
          f"| undifferentiated TPR: {baseline['undifferentiated_tpr_acq_mod']:.3f} -> "
          f"{r['undifferentiated_tpr_acq_mod']:.3f}\n")

Baseline (no reject): hard_mean=0.9724 | acquisition_mean=0.9746

tau=0.5: Δhard_mean=-0.0023 | Δacquisition_mean=-0.0020 | rigettati(acq,mod)=0.4% -> PASS (entro il floor)
   clothing TPR (attrattore, acq. moderate): 0.998 -> 0.997 | undifferentiated TPR: 0.950 -> 0.964

tau=0.7: Δhard_mean=-0.0123 | Δacquisition_mean=-0.0132 | rigettati(acq,mod)=2.6% -> FAIL (peggiora oltre il floor pre-registrato)
   clothing TPR (attrattore, acq. moderate): 0.998 -> 0.994 | undifferentiated TPR: 0.950 -> 0.971

